# Tool check: stochastic

Runs each tool of this capability **directly, without the agent**, on the inputs you give. One section per tool: run the setup, then only the section you need.

In [ ]:
import os, sys, time, json
ROOT = os.path.abspath(os.path.join("..", ".."))      # notebooks/tools/ -> project root
sys.path.insert(0, ROOT)
from dynalyst.orchestration import tools as registry
from IPython.display import Image, display


def arguments(tool):
    """The tool's arguments, read from the tool itself."""
    schema = registry.schema_of(registry.all_tools()[tool])
    print(tool, "-- required:", schema.get("required", []))
    for name, spec in schema.get("properties", {}).items():
        print(f"  {name} ({spec.get('type', 'any')}): {(spec.get('description') or '')[:150]}")


def run_tool(tool, args, pasted=""):
    """Run one tool alone: your ARGS, or the agent's call pasted from its log."""
    if pasted.strip():
        args = json.loads(pasted)
    print(tool, args)
    t = time.time()
    raw = registry.execute(tool, args)
    return show(tool, raw, time.time() - t)


def show(tool, raw, seconds):
    """The result: error first, then each field (long ones shortened), then plots."""
    print(f"{tool} ran in {seconds:.1f} s")
    try:
        data = json.loads(raw)
    except (TypeError, ValueError):
        print(str(raw)[:3000])
        return None
    if isinstance(data, dict):
        if data.get("error"):
            print("ERROR:", data["error"])
        for key, value in data.items():
            text = json.dumps(value, default=str)
            print(f"- {key}: {text[:400]}{' ...' if len(text) > 400 else ''}")
        for key, value in data.items():
            if isinstance(value, str) and value.lower().endswith(".png"):
                path = value if os.path.isabs(value) else os.path.join(ROOT, value)
                if os.path.exists(path):
                    print(key)
                    display(Image(filename=path))
    else:
        print(json.dumps(data, default=str)[:3000])
    return data

TOOLS = ['stochastic_simulation', 'propensity_table']

## `stochastic_simulation`


In [ ]:
arguments("stochastic_simulation")
# Fill in what you need (required: []), OR paste the agent's call from its log
# (the {...} after "SELECT stochastic_simulation(") into PASTED.
ARGS = {
    # "model_name_or_path": ...,   # string: SBML file / name, XPP .ode/.txt file, or BioModels ID. Leave out when typing ODEs.
    # "species": ...,   # array: Typed-ODE models only: species names in order.
    # "odes": ...,   # object: Typed-ODE models only: {species: right-hand side}.
    # "parameters": ...,   # object: {name: value}: changed parameter values for this run (for typed ODEs: their values).
    # "t_end": ...,   # number: End time (model time units).
    # "omega": ...,   # array: System size(s) Omega, molecules per model state unit: pass it only when the task names one, or sever
    # "methods": ...,   # array: Default ['ssa']. Use ['cle'] for CLE only; ['ssa','cle'] only when both are requested.
    # "n_runs": ...,   # integer: Stochastic runs per method and omega (default 20).
    # "n_points": ...,   # integer: Output time points (default 400).
    # "dt": ...,   # number: CLE time step (default t_end / (20 n_points)).
    # "seed": ...,   # integer: 
    # "initial_conditions": ...,   # object: {species: initial value in the model's state units}
    # "plot_species": ...,   # array: Species to plot (default the first four).
    # "export_matlab": ...,   # boolean: Write MATLAB code for the selected methods. Tau-leaping export is unsupported.
    # "record_dt": ...,   # number: Sampling interval of the recorded time course (sets n_points).
    # "burn_in": ...,   # number: Time discarded before pooling the distribution (default 10% of t_end).
    # "distribution": ...,   # boolean: Empirical pooled distribution (default true); peaks do not establish stationarity or deterministic s
    # "distribution_species": ...,   # array: Read-out species for the distribution (default: the XPP autovar, else the most variable species).
    # "start_in_each_state": ...,   # boolean: When the ODE has several stable states but every run stayed in one, also start runs in each stable s
    # "ssa_method": ...,   # string: auto: exact SSA (numpy when it fits the time budget, else JAX if installed, else skipped with an est
}
PASTED = ""
result = run_tool("stochastic_simulation", ARGS, PASTED)

## `propensity_table`


In [ ]:
arguments("propensity_table")
# Fill in what you need (required: []), OR paste the agent's call from its log
# (the {...} after "SELECT propensity_table(") into PASTED.
ARGS = {
    # "model_name_or_path": ...,   # string: SBML file / name, XPP .ode/.txt file, or BioModels ID. Leave out when typing ODEs.
    # "species": ...,   # array: Typed-ODE models only: species names in order.
    # "odes": ...,   # object: Typed-ODE models only: {species: right-hand side}.
    # "parameters": ...,   # object: {name: value}: changed parameter values for this run (for typed ODEs: their values).
    # "omega": ...,   # number: Omega for the MATLAB script (default 1000).
    # "export_matlab": ...,   # boolean: 
}
PASTED = ""
result = run_tool("propensity_table", ARGS, PASTED)

## The agent on the same request (optional)

In [ ]:
# Optional: ask the agent the same thing and compare (needs the API key).
QUERY = ""                       # e.g. the request that went wrong
if QUERY.strip():
    from dynalyst.agents.unified_agent import run
    t = time.time()
    rec = run(QUERY)
    print(f"agent: {time.time() - t:.1f} s\n")
    calls = [s for s in rec.steps if s.get("tool") in TOOLS]
    print(f"the agent made {len(calls)} call(s) to these tools:")
    for a in calls:
        print(" ", a.get("tool"), json.dumps(a.get("arguments"), default=str)[:400],
              "|", a.get("status"), (a.get("error") or "")[:200])
    print("\nanswer:\n", rec.final_answer[:2000])

**How to read it**
- The tool fails or gives a wrong result here, with the agent's arguments: the **tool** is the cause.
- The tool works here, but the chat went wrong: the **agent** is the cause (wrong arguments,
  wrong tool, or a wrong reading of the result). Compare with the agent's calls above.
- The tool works here only with different arguments: the agent passed wrong arguments.